# 05 — Univariate vs Multivariate

apakah AOD + cuaca benar-benar membantu prediksi PM2.5?

- **Model 1 (Univariate):** input = `[PM2.5]` saja
- **Model 2 (Multivariate):** input = `[temp, dew, humidity, precip, windspeed, AOD, PM2.5]`

Kedua model menggunakan parameter hasil notebook 03/04 (Adam, lr=1e-3, units=32, dropout=0.1) untuk fair comparison.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd

from src import config as C
from src.data_loader import load_all_stations
from src.evaluation import compute_metrics
from src.model import build_lstm, set_seed, train_model
from src.preprocessing import build_pipeline, inverse_target

In [2]:
FIXED_PARAMS = {'lstm_units': 32, 'dropout_rate': 0.1, 'optimizer': 'adam', 'learning_rate': 1e-3}
EPOCHS = 100
PATIENCE = 15
STATIONS_TO_RUN = C.HEALTHY_STATIONS

FEATURES_UNI = [C.TARGET]
FEATURES_MULTI = C.WEATHER_COLS + [C.AOD_COL, C.TARGET]

dfs = load_all_stations(reindex=True)

In [3]:
def run(features, df_station):
    data = build_pipeline(df_station, features, smooth_aod=False)
    set_seed()
    n_features = data['X_train'].shape[2]
    model = build_lstm(input_shape=(C.LOOKBACK, n_features), **FIXED_PARAMS)
    train_model(model, data['X_train'], data['y_train'], data['X_val'], data['y_val'],
                epochs=EPOCHS, patience=PATIENCE, verbose=0)

    y_pred = model.predict(data['X_test'], verbose=0).flatten()
    y_pred = inverse_target(y_pred, data['scaler'], data['target_idx'], n_features)
    y_true = inverse_target(data['y_test'], data['scaler'], data['target_idx'], n_features)
    return compute_metrics(y_true, y_pred)

In [4]:
rows = []
for s in STATIONS_TO_RUN:
    print(s, '...')
    m_uni   = run(FEATURES_UNI,   dfs[s])
    m_multi = run(FEATURES_MULTI, dfs[s])
    rows.append({
        'station': s,
        'R2_uni':   m_uni['R2'],   'RMSE_uni':   m_uni['RMSE'],   'MAE_uni':   m_uni['MAE'],
        'R2_multi': m_multi['R2'], 'RMSE_multi': m_multi['RMSE'], 'MAE_multi': m_multi['MAE'],
    })

df_cmp = pd.DataFrame(rows)
df_cmp['delta_R2'] = df_cmp['R2_multi'] - df_cmp['R2_uni']
df_cmp.to_csv(C.METRICS_DIR / '05_univariate_vs_multivariate.csv', index=False)
df_cmp

us_embassy_1 ...
us_embassy_2 ...
bundaran_hi ...
kelapa_gading ...
jagakarsa ...
lubang_buaya ...
kebun_jeruk ...


,station,R2_uni,RMSE_uni,MAE_uni,R2_multi,RMSE_multi,MAE_multi,delta_R2
0,us_embassy_1,0.624077,12.928986,10.010865,-5.131795,52.216647,44.409932,-5.755872
1,us_embassy_2,0.551717,16.973561,12.771772,0.353329,20.386300,14.990627,-0.198388
2,bundaran_hi,0.609391,18.766586,14.001966,0.591993,19.179960,14.654796,-0.017398
3,kelapa_gading,0.674833,13.126060,9.999037,0.635489,13.897503,10.827941,-0.039344
4,jagakarsa,0.459605,13.541451,9.057799,0.393457,14.346315,10.264165,-0.066148
5,lubang_buaya,0.353632,19.793256,13.653332,-0.283602,27.892821,23.409972,-0.637234
6,kebun_jeruk,0.687660,17.242958,11.874909,0.461190,22.647274,18.099090,-0.226470


**Interpretasi:** kolom `delta_R2 > 0` berarti AOD+cuaca menambah skill model. `delta_R2 ≈ 0` atau negatif berarti PM2.5 sendiri (autoregressive) sudah cukup informatif.